


NumPy gives you fast arrays of numbers, but real world data isn't just numbers it's rows with column names, mixed types, missing values, dates, categories. Pandas is built on top of NumPy and gives you a spreadsheet like structure (the DataFrame) with all the tools to clean, explore, and reshape that messy real-world data before it ever reaches a model.


1. Why Pandas?
2. Series & DataFrame :the core objects
3. Loading real data (read_csv)
4. Selecting & filtering data
5. Handling missing data
6. GroupBy & aggregation
7. Feature engineering basics
8. Live demo: raw data → model-ready data
9. Wrap-up: Pandas + NumPy + your ML model


## 1. Why Pandas?

Imagine a dataset of 1000 house listings  price, size, location, number of bedrooms, some missing values, some text columns. Try doing this in plain NumPy:
- Every column has to be a separate array (no column *names*)
- Missing values are painful to handle
- No easy "group by neighborhood and get average price"

Pandas solves exactly this. It's the tool literally every data scientist opens first, before any model is even mentioned.


In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)


Pandas version: 3.0.0


## 2. Series & DataFrame :The Core Objects

- **Series** = a single labeled column (like one column in a spreadsheet)
- **DataFrame** = a full table made of many Series sharing the same row labels (index)


In [2]:
# Series :1D labeled array
prices = pd.Series([250000, 310000, 180000, 275000], name="price")
print("Series:\n", prices)
print("\nSeries index:", prices.index.tolist())

# DataFrame : 2D table
data = {
    "house_id": [1, 2, 3, 4],
    "size_sqft": [1200, 1500, 900, 1350],
    "bedrooms": [3, 4, 2, 3],
    "price": [250000, 310000, 180000, 275000],
    "location": ["Downtown", "Suburb", "Downtown", "Suburb"]
}
df = pd.DataFrame(data)
print("\nDataFrame:\n", df)

print("\n Key inspection methods ")
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nHead (first rows):\n", df.head(2))
print("\nQuick stats:\n", df.describe())


Series:
 0    250000
1    310000
2    180000
3    275000
Name: price, dtype: int64

Series index: [0, 1, 2, 3]

DataFrame:
    house_id  size_sqft  bedrooms   price  location
0         1       1200         3  250000  Downtown
1         2       1500         4  310000    Suburb
2         3        900         2  180000  Downtown
3         4       1350         3  275000    Suburb

 Key inspection methods 
Shape: (4, 5)

Column names: ['house_id', 'size_sqft', 'bedrooms', 'price', 'location']

Data types:
 house_id     int64
size_sqft    int64
bedrooms     int64
price        int64
location       str
dtype: object

Head (first rows):
    house_id  size_sqft  bedrooms   price  location
0         1       1200         3  250000  Downtown
1         2       1500         4  310000    Suburb

Quick stats:
        house_id    size_sqft  bedrooms          price
count  4.000000     4.000000  4.000000       4.000000
mean   2.500000  1237.500000  3.000000  253750.000000
std    1.290994   256.173769  0.8

Why this matters for ML: every dataset you'll load for a model Titanic, house prices, customer churn starts life as a DataFrame. .head(), `.info(), .describe() are the first three commands you run on any new dataset, always.


## 3. Loading Real Data read_csv()

In real projects, you almost never type data by hand  you load it from a file.


In [3]:
# We'll simulate this by writing a small CSV to disk, then loading it exactly
# the way you would with a real dataset file.

csv_text = '''student_id,name,study_hours,attendance_pct,previous_score,passed
1,Amit,5,90,72,1
2,Sita,2,60,45,0
3,Rita,8,95,88,1
4,Bikash,,70,,0
5,Puja,6,,79,1
6,Kiran,1,55,,0
7,Sunita,7,88,85,1
8,Rohan,3,65,58,0
'''

with open("students.csv", "w") as f:
    f.write(csv_text)

df = pd.read_csv("students.csv")
print(df)
print("\nShape:", df.shape)
print("\nInfo:")
df.info()


   student_id    name  study_hours  attendance_pct  previous_score  passed
0           1    Amit          5.0            90.0            72.0       1
1           2    Sita          2.0            60.0            45.0       0
2           3    Rita          8.0            95.0            88.0       1
3           4  Bikash          NaN            70.0             NaN       0
4           5    Puja          6.0             NaN            79.0       1
5           6   Kiran          1.0            55.0             NaN       0
6           7  Sunita          7.0            88.0            85.0       1
7           8   Rohan          3.0            65.0            58.0       0

Shape: (8, 6)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   student_id      8 non-null      int64  
 1   name            8 non-null      str    
 2   study_hours     7 non-null   

Notice: some cells already show NaN (missing values) this is completely normal in real data and exactly what we'll clean up in section 5.

Why this matters for ML:read_csv() is the single most-used line of code in any ML project's first cell. Pandas also has read_excel(), read_json(), read_sql() for other formats.


## 4. Selecting & Filtering Data

Just like NumPy's boolean masking, but now with column names and labels.


In [4]:
# Selecting a single column : returns a Series
print("Study hours column:\n", df["study_hours"])

# Selecting multiple columns : returns a DataFrame
print("\nName + score:\n", df[["name", "previous_score"]])

# .loc : select by LABEL (row label, column label)
print("\nRow with student_id label 2 (loc):\n", df.loc[1])

# .iloc : select by INTEGER POSITION
print("\nFirst row (iloc):\n", df.iloc[0])

# Boolean filtering :the Pandas equivalent of NumPy masking
print("\n--- Students who passed ---")
print(df[df["passed"] == 1])

print("\n--- Students who studied more than 5 hours ---")
print(df[df["study_hours"] > 5])

# Combining conditions
print("\n--- Studied > 4 hours AND passed ---")
print(df[(df["study_hours"] > 4) & (df["passed"] == 1)])


Study hours column:
 0    5.0
1    2.0
2    8.0
3    NaN
4    6.0
5    1.0
6    7.0
7    3.0
Name: study_hours, dtype: float64

Name + score:
      name  previous_score
0    Amit            72.0
1    Sita            45.0
2    Rita            88.0
3  Bikash             NaN
4    Puja            79.0
5   Kiran             NaN
6  Sunita            85.0
7   Rohan            58.0

Row with student_id label 2 (loc):
 student_id           2
name              Sita
study_hours        2.0
attendance_pct    60.0
previous_score    45.0
passed               0
Name: 1, dtype: object

First row (iloc):
 student_id           1
name              Amit
study_hours        5.0
attendance_pct    90.0
previous_score    72.0
passed               1
Name: 0, dtype: object

--- Students who passed ---
   student_id    name  study_hours  attendance_pct  previous_score  passed
0           1    Amit          5.0            90.0            72.0       1
2           3    Rita          8.0            95.0            88.

**Why this matters for ML:** filtering rows like "keep only valid records" or "separate passed vs failed students to compare features" is something you'll write in nearly every notebook, especially during exploratory data analysis (EDA).


 Handling Missing Data

Models generally **cannot** handle NaN values this step is non-negotiable before training.


In [5]:
print("Missing values per column:\n", df.isnull().sum())

# Option 1: Drop rows with any missing values
df_dropped = df.dropna()
print("\nAfter dropna() - shape:", df_dropped.shape)

# Option 2: Fill missing values 
df_filled = df.copy()
df_filled["study_hours"] = df_filled["study_hours"].fillna(df_filled["study_hours"].mean())
df_filled["attendance_pct"] = df_filled["attendance_pct"].fillna(df_filled["attendance_pct"].median())
df_filled["previous_score"] = df_filled["previous_score"].fillna(df_filled["previous_score"].mean())

print("\nAfter filling with mean/median:\n", df_filled)
print("\nMissing values now:\n", df_filled.isnull().sum())


Missing values per column:
 student_id        0
name              0
study_hours       1
attendance_pct    1
previous_score    2
passed            0
dtype: int64

After dropna() - shape: (5, 6)

After filling with mean/median:
    student_id    name  study_hours  attendance_pct  previous_score  passed
0           1    Amit     5.000000            90.0       72.000000       1
1           2    Sita     2.000000            60.0       45.000000       0
2           3    Rita     8.000000            95.0       88.000000       1
3           4  Bikash     4.571429            70.0       71.166667       0
4           5    Puja     6.000000            70.0       79.000000       1
5           6   Kiran     1.000000            55.0       71.166667       0
6           7  Sunita     7.000000            88.0       85.000000       1
7           8   Rohan     3.000000            65.0       58.000000       0

Missing values now:
 student_id        0
name              0
study_hours       0
attendance_pct  

**Why this matters for ML:** this is a genuine make-or-break step. Drop too aggressively and you lose valuable data; fill carelessly and you introduce bias. Real preprocessing pipelines (like Scikit-learn's `SimpleImputer`) do exactly this kind of fill, just wrapped in a reusable object.


 GroupBy & Aggregation

This is how you find patterns in data *before* you ever train a model  a huge part of exploratory data analysis (EDA).


In [6]:
# Add a categorical column to group by
df_filled["pass_label"] = df_filled["passed"].map({1: "Passed", 0: "Failed"})

print(" Average stats by pass/fail ")
print(df_filled.groupby("pass_label")[["study_hours", "attendance_pct", "previous_score"]].mean())

print("\n Count of students per group ")
print(df_filled.groupby("pass_label").size())

print("\n Multiple aggregations at once ")
print(df_filled.groupby("pass_label")["study_hours"].agg(["mean", "min", "max", "count"]))


 Average stats by pass/fail 
            study_hours  attendance_pct  previous_score
pass_label                                             
Failed         2.642857           62.50       61.333333
Passed         6.500000           85.75       81.000000

 Count of students per group 
pass_label
Failed    4
Passed    4
dtype: int64

 Multiple aggregations at once 
                mean  min       max  count
pass_label                                
Failed      2.642857  1.0  4.571429      4
Passed      6.500000  5.0  8.000000      4


Why this matters for ML: groupby is how you notice things like "students who passed studied 3x more hours on average"  an insight that tells you study_hours will likely be an important feature for your model, before you've trained anything.


 Feature Engineering Basics

Creating new, more useful columns from existing ones  often the difference between an average model and a great one.


In [7]:
# Creating a new column with simple math
df_filled["hours_per_attendance"] = df_filled["study_hours"] / (df_filled["attendance_pct"] / 100)

# Mapping categories to numbers - models need numeric input
df_filled["passed_numeric"] = df_filled["passed"]  # already numeric here, but shown for pattern

# .apply() :run a custom function on every value/row
def score_category(score):
    if score >= 80:
        return "High"
    elif score >= 60:
        return "Medium"
    else:
        return "Low"

df_filled["score_band"] = df_filled["previous_score"].apply(score_category)

print(df_filled[["name", "study_hours", "attendance_pct", "hours_per_attendance",
                  "previous_score", "score_band"]])


     name  study_hours  attendance_pct  hours_per_attendance  previous_score  \
0    Amit     5.000000            90.0              5.555556       72.000000   
1    Sita     2.000000            60.0              3.333333       45.000000   
2    Rita     8.000000            95.0              8.421053       88.000000   
3  Bikash     4.571429            70.0              6.530612       71.166667   
4    Puja     6.000000            70.0              8.571429       79.000000   
5   Kiran     1.000000            55.0              1.818182       71.166667   
6  Sunita     7.000000            88.0              7.954545       85.000000   
7   Rohan     3.000000            65.0              4.615385       58.000000   

  score_band  
0     Medium  
1        Low  
2       High  
3     Medium  
4     Medium  
5     Medium  
6       High  
7        Low  


Why this matters for ML: models don't understand text categories like "Downtown" or "High" this is the step where you either engineer smarter numeric features or prepare categories for encoding (e.g. pd.get_dummies() for one-hot encoding).


 Live Demo: Raw Data → Model-Ready Data

Let's put it all together take the raw messy CSV and walk it all the way to something you could hand straight to a Scikit-learn model.


In [8]:
# Step 1: Load raw data
raw = pd.read_csv("students.csv")
print("Step 1 - Raw data:\n", raw)

# Step 2: Handle missing values
clean = raw.copy()
for col in ["study_hours", "attendance_pct", "previous_score"]:
    clean[col] = clean[col].fillna(clean[col].mean())
print("\nStep 2 - Missing values handled")

# Step 3: Feature engineering
clean["hours_per_attendance"] = clean["study_hours"] / (clean["attendance_pct"] / 100)
print("Step 3 - New feature added")

# Step 4: Select final features + target, drop ID column
features = clean[["study_hours", "attendance_pct", "previous_score", "hours_per_attendance"]]
target = clean["passed"]
print("\nStep 4 - Final features:\n", features)
print("\nTarget:\n", target.tolist())

# Step 5: Convert to NumPy - this is the handoff point to any ML model
X = features.to_numpy()   # or features.values
y = target.to_numpy()
print("\nStep 5 - As NumPy arrays, ready for a model:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nX:\n", X)


Step 1 - Raw data:
    student_id    name  study_hours  attendance_pct  previous_score  passed
0           1    Amit          5.0            90.0            72.0       1
1           2    Sita          2.0            60.0            45.0       0
2           3    Rita          8.0            95.0            88.0       1
3           4  Bikash          NaN            70.0             NaN       0
4           5    Puja          6.0             NaN            79.0       1
5           6   Kiran          1.0            55.0             NaN       0
6           7  Sunita          7.0            88.0            85.0       1
7           8   Rohan          3.0            65.0            58.0       0

Step 2 - Missing values handled
Step 3 - New feature added

Step 4 - Final features:
    study_hours  attendance_pct  previous_score  hours_per_attendance
0     5.000000       90.000000       72.000000              5.555556
1     2.000000       60.000000       45.000000              3.333333
2     8.000

**This is the full pipeline** you'll repeat in almost every ML project: `read_csv` → clean → engineer features → `.to_numpy()` → feed into `model.fit(X, y)`.


 Wrap-Up : Pandas + NumPy + Your Model

3 things to remember:
1. DataFrame = your main tool for loading, cleaning, and exploring real-world tabular data.
2. Missing data and feature engineering happen in Pandas, before anything touches a model.
3. .to_numpy() / .valuesis the bridge — it converts your cleaned DataFrame into the raw NumPy array that Scikit-learn, TensorFlow, or PyTorch actually expect.


CSV file  →  pandas.read_csv()  →  clean & engineer in DataFrame  →  .to_numpy()  →  model.fit(X, y)



